### OCI Data Science - Useful Tips
<details>
<summary><font size="2">Check for Public Internet Access</font></summary>

```python
import requests
response = requests.get("https://oracle.com")
assert response.status_code==200, "Internet connection failed"
```
</details>
<details>
<summary><font size="2">Helpful Documentation </font></summary>
<ul><li><a href="https://docs.cloud.oracle.com/en-us/iaas/data-science/using/data-science.htm">Data Science Service Documentation</a></li>
<li><a href="https://docs.cloud.oracle.com/iaas/tools/ads-sdk/latest/index.html">ADS documentation</a></li>
</ul>
</details>
<details>
<summary><font size="2">Typical Cell Imports and Settings for ADS</font></summary>

```python
%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(format='%(levelname)s:%(message)s', level=logging.ERROR)

import ads
from ads.dataset.factory import DatasetFactory
from ads.automl.provider import OracleAutoMLProvider
from ads.automl.driver import AutoML
from ads.evaluations.evaluator import ADSEvaluator
from ads.common.data import ADSData
from ads.explanations.explainer import ADSExplainer
from ads.explanations.mlx_global_explainer import MLXGlobalExplainer
from ads.explanations.mlx_local_explainer import MLXLocalExplainer
from ads.catalog.model import ModelCatalog
from ads.common.model_artifact import ModelArtifact
```
</details>
<details>
<summary><font size="2">Useful Environment Variables</font></summary>

```python
import os
print(os.environ["NB_SESSION_COMPARTMENT_OCID"])
print(os.environ["PROJECT_OCID"])
print(os.environ["USER_OCID"])
print(os.environ["TENANCY_OCID"])
print(os.environ["NB_REGION"])
```
</details>

In [74]:
from oci.config import from_file
import base64
import json
import os
from os import path
from oci.vault.models import SecretContentDetails,Base64SecretContentDetails,CreateSecretDetails,Secret
from oci.vault import VaultsClientCompositeOperations,VaultsClient
from oci.secrets import SecretsClient

In [42]:
ads.set_auth(auth='resource_principal')

# 1. A vault is already created

# 2. Creating the secrets

In [43]:
credentials={
    'database': 'datamart',
    'username':'test',
    'password': 'tester' 
}

# 3. Encoding the Secrets

## 3.1 Encode the Secrets into Base64

In [44]:
"""
A helper function that encodes the JSON version of a secrets dictionary into base64
"""
def dict_to_secret(dictionary):
    return base64.b64encode(json.dumps(dictionary).encode('ascii')).decode('ascii')

## 3.2 Encode the Secrets using OCI

In [45]:
"""
Encoding the secrets using the OCI Base64SecretContentDetails class
"""

secret_content_details=Base64SecretContentDetails(
    content_type=SecretContentDetails.CONTENT_TYPE_BASE64,
    stage=SecretContentDetails.STAGE_CURRENT,
    content=dict_to_secret(credentials)
)

## 3.3 Bundle the secret at the metadata

In [65]:
"""
Bundle the secret at its metadata
"""

secrets_details=CreateSecretDetails(
    compartment_id=os.environ["TENANCY_OCID"],
    description= 'Storing Sample Secrets to the Vault',
    secret_content=secret_content_details,
    secret_name='Database_Credentials',
    vault_id='ocid1.vault.oc1.us-chicago-1.ijurdkywaaeje.abxxeljriof5l77jq7aiag2sxli4jrwcwg7vwjbwhuvrybewxdjt6uaid46q',
    key_id='ocid1.key.oc1.us-chicago-1.ijurdkywaaeje.abxxeljt3dystiblybvv6r3gutcq4q6qqyglcsltjjgte6wbnqfvefn2akja'
)

# 4. Storing the Secrets to the Vault

In [66]:
config=from_file(path.join(path.expanduser('~'),'.oci','config'),'DEFAULT')
vaults_client_composite=VaultsClientCompositeOperations(VaultsClient(config))

secret=vaults_client_composite.create_secret_and_wait_for_state(create_secret_details=secrets_details,wait_for_states=[Secret.LIFECYCLE_STATE_ACTIVE]).data

# 5. Retrieving the Secrets

## 5.1 Decode the Based64 version of the secret as a dictionary

In [70]:
"""
Retrieving the base64 version of the stored secret as a dictionary
"""

def secret_to_dict(wallet):
    return json.loads(base64.b64decode(wallet.encode('ascii')).decode('ascii'))

## 5.2 Get a Secret Bundle using the Secret's OCID

In [76]:
secret_bundle=SecretsClient(config).get_secret_bundle('ocid1.vaultsecret.oc1.us-chicago-1.amaaaaaabnt4rwiac7aopjazvhbgog54mgo4lv22plbodhpxwywn2ih5euyq')

## 5.3 Get the decoded secret as a dictionary

In [77]:
secret_content=secret_to_dict(secret_bundle.data.secret_bundle_content.content)

# 5.4 View the Secret

In [78]:
print(secret_content)

{'database': 'datamart', 'username': 'test', 'password': 'tester'}
